In [48]:
import sys
import os
from unittest.mock import MagicMock

#----------------------------------------------
# FIXING ENVIRONMENT ISSUE IN WINDOWS
#----------------------------------------------

project_root = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\matcha-tts"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

for root, dirs, files in os.walk(os.path.join(project_root, "matcha")):
    if "__init__.py" not in files:
        with open(os.path.join(root, "__init__.py"), "w") as f:
            pass
        print(f"🔧 Created missing __init__.py in: {root}")

matcha_mock = MagicMock()
utils_mock = MagicMock()
mas_mock = MagicMock()

sys.modules["matcha.utils.monotonic_align"] = mas_mock
sys.modules["matcha.utils.monotonic_align.core"] = mas_mock

try:
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from IPython.display import Audio
    
    import matcha
    import matcha.utils
    import matcha.models
    
    from matcha.models.matcha_tts import MatchaTTS
    from matcha.text import text_to_sequence
    from matcha.utils.utils import intersperse
    
    print("🚀 SUCCESS! Model and utils imported successfully.")
except Exception as e:
    print(f"❌ Still failing: {e}")
    if 'matcha' in sys.modules:
        print(f"Matcha location in memory: {sys.modules['matcha'].__file__}")

os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = r'C:\Program Files\eSpeak NG\libespeak-ng.dll'

def denormalize(mel, mean, std):
    return (mel * std) + mean

🔧 Created missing __init__.py in: C:\Users\alvar\Code\Github\MedEmoji-TTS\matcha-tts\matcha\hifigan\__pycache__
🚀 SUCCESS! Model and utils imported successfully.


# Fine-tuned Model

In [49]:
import math
import torch
import math
from matcha.models.matcha_tts import MatchaTTS

CHECKPOINT_PATH = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\model\anger-v2"
MEL_MEAN = -5.536622
MEL_STD = 2.116101

# 1. Load the model
model = MatchaTTS.load_from_checkpoint(
    CHECKPOINT_PATH, 
    map_location="cpu", 
    strict=False,
    n_vocab=198, 
    n_spks=1, 
    spk_emb_dim=0,
    weights_only=False
)

# 2. Fix the Loops (n_layers, n_heads)
def fix_loops(module):
    for attr in ['n_layers', 'n_heads', 'kernel_size']:
        if hasattr(module, attr):
            val = getattr(module, attr)
            if isinstance(val, (list, tuple)):
                setattr(module, attr, int(val[0]))
    if hasattr(module, 'children'):
        for child in module.children():
            fix_loops(child)

fix_loops(model)

# 3. Fix LayerNorm and ensure its 'channels' is a tuple of one integer
for m in model.modules():
    if 'LayerNorm' in m.__class__.__name__:
        if hasattr(m, 'channels'):
            if isinstance(m.channels, int):
                m.channels = (m.channels,)
            elif isinstance(m.channels, list):
                m.channels = tuple(m.channels)

# Matcha-TTS expects these as (1, 80, 1) tensors.
model.mel_mean = torch.full((1, 80, 1), MEL_MEAN)
model.mel_std = torch.full((1, 80, 1), MEL_STD)

# Ensure the encoder knows about the features
model.n_feats = 80
model.encoder.n_feats = 80

model.eval()

MatchaTTS(
  (encoder): TextEncoder(
    (emb): Embedding(198, 192)
    (prenet): ConvReluNorm(
      (conv_layers): ModuleList(
        (0-2): 3 x Conv1d(192, 192, kernel_size=5, stride=(1,), padding=(2,))
      )
      (norm_layers): ModuleList(
        (0-2): 3 x LayerNorm()
      )
      (relu_drop): Sequential(
        (0): ReLU()
        (1): Dropout(p=0.5, inplace=False)
      )
      (proj): Conv1d(192, 192, kernel_size=1, stride=(1,))
    )
    (encoder): Encoder(
      (drop): Dropout(p=0.1, inplace=False)
      (attn_layers): ModuleList(
        (0-5): 6 x MultiHeadAttention(
          (conv_q): Conv1d(192, 192, kernel_size=1, stride=(1,))
          (conv_k): Conv1d(192, 192, kernel_size=1, stride=(1,))
          (conv_v): Conv1d(192, 192, kernel_size=1, stride=(1,))
          (query_rotary_pe): RotaryPositionalEmbeddings()
          (key_rotary_pe): RotaryPositionalEmbeddings()
          (conv_o): Conv1d(192, 192, kernel_size=1, stride=(1,))
          (drop): Dropout(p=0.1,

In [50]:
from matcha.text.symbols import symbols

_symbol_to_id = {s: i for i, s in enumerate(symbols)}

def speak(text, temperature=0.667, length_scale=1.0):
    try:
        text = text.lower()
        # Direct character to ID mapping
        raw_seq = [_symbol_to_id.get(s, 1) for s in text]
        inter_seq = intersperse(raw_seq, 0)
        
        x = torch.LongTensor(inter_seq).unsqueeze(0)
        x_lengths = torch.LongTensor([x.shape[1]])
        
        with torch.no_grad():
            outputs = model.synthesise(
                x, x_lengths, 
                n_timesteps=50, 
                temperature=temperature, 
                spks=None, 
                length_scale=length_scale
            )
        
        mel = (outputs['decoder_outputs'] * model.mel_std) + model.mel_mean
        return mel.cpu().numpy()
        
    except Exception as e:
        print(f"❌ Synthesis failed: {e}")
        return None

In [51]:
import torch

# Use the maintained mirror. 'hifigan_lj_v1' is exactly the LJSpeech V1 model.
# trust_repo=True is required by newer versions of PyTorch.
vocoder = torch.hub.load("lars76/bigvgan-mirror", "hifigan_lj_v1", trust_repo=True)

vocoder.eval()
vocoder.to('cpu') # Or 'cuda' if you have a GPU working!
print("🎸 HiFi-GAN is tuned and ready to play!")

Using cache found in C:\Users\alvar/.cache\torch\hub\lars76_bigvgan-mirror_main


🎸 HiFi-GAN is tuned and ready to play!


In [52]:
from matcha.text.symbols import symbols
from matcha.text import text_to_sequence

test_text = "I am absolutely finished with these architecture errors!"
cleaner = ['english_cleaners2']

# 1. Generate the sequence tuple
sequence_tuple = text_to_sequence(test_text, cleaner)

# 2. Extract the actual IDs (the first element)
id_list = sequence_tuple[0] 

print(f"📝 Raw Text: {test_text}")
print(f"🔢 Sequence IDs: {id_list}")
print(f"📏 True Length: {len(id_list)} symbols")

# 3. Decode using the symbols dictionary
decoded = [symbols[i] for i in id_list if i < len(symbols)]
print(f"🗣️ What the model 'hears': {''.join(decoded)}")

📝 Raw Text: I am absolutely finished with these architecture errors!
🔢 Sequence IDs: [156, 43, 102, 16, 72, 55, 16, 157, 72, 44, 61, 83, 54, 156, 63, 158, 62, 54, 51, 16, 48, 156, 102, 56, 102, 131, 62, 16, 65, 102, 81, 16, 81, 51, 158, 68, 16, 156, 69, 158, 123, 53, 102, 62, 157, 86, 53, 62, 131, 85, 123, 16, 156, 86, 123, 85, 68, 5]
📏 True Length: 58 symbols
🗣️ What the model 'hears': ˈaɪ æm ˌæbsəlˈuːtli fˈɪnɪʃt wɪð ðiːz ˈɑːɹkɪtˌɛktʃɚɹ ˈɛɹɚz!


In [53]:
# 1. Synthesize with "Angry" settings
# Temperature 0.1: Makes the model very 'sure' of its sounds (less breathy)
# length_scale 0.8: Speeds it up. Angry people talk faster!
mel_output = speak(test_text, temperature=0.1, length_scale=1)

# 2. Convert with HiFi-GAN
with torch.no_grad():
    audio = vocoder(torch.FloatTensor(mel_output))

# 3. Play the result
display(Audio(audio.squeeze().numpy(), rate=22050))

# Model Comparison

In [60]:
import torch

# 1. Update these paths to your local Windows paths
BASE_MODEL_PATH = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\model\matcha_ljspeech.ckpt"
CHECKPOINT_PATH = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\model\anger-v2.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cpu", 
        strict=False,
        n_vocab=vocab_size, # This must be specific to the file!
        n_spks=1, 
        spk_emb_dim=0
    )
    fix_loops(m)
    m.mel_mean = torch.full((1, 80, 1), -5.51) 
    m.mel_std = torch.full((1, 80, 1), 2.11)
    m.eval()
    return m

# 2. LOAD BOTH (Using the specific sizes the error told us)
model_neutral = load_matcha(BASE_MODEL_PATH, "Neutral Model", 178)
model_angry = load_matcha(CHECKPOINT_PATH, "Angry Model", 198)

print("🚀 Both models loaded and ready for comparison!")

📦 Loading Neutral Model (Vocab: 178)...
📦 Loading Angry Model (Vocab: 198)...
🚀 Both models loaded and ready for comparison!


In [38]:
import torch
import os
from IPython.display import display, Audio

# --- 1. THE REPAIRED LOAD FUNCTION ---

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab size: {vocab_size})...")
    
    # We pass the specific vocab_size needed for THIS checkpoint
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cpu", 
        strict=False,
        n_vocab=vocab_size, 
        n_spks=1, 
        spk_emb_dim=0
    )
    
    # Re-apply the Windows/MRes hacks to bypass compiled C++ modules
    fix_loops(m) 
    
    # Standardize statistics for a fair baseline comparison
    m.mel_mean = torch.full((1, 80, 1), -5.51) 
    m.mel_std = torch.full((1, 80, 1), 2.11)
    
    m.eval()
    return m

# --- 2. LOAD BOTH MODELS WITH THEIR CORRECT SIZES ---

# Neutral base is 178, your fine-tuned is 198
model_neutral = load_matcha(BASE_MODEL_PATH, "Neutral Model", 178)
model_angry = load_matcha(CHECKPOINT_PATH, "Fine-tuned Angry Model", 198)

# --- 3. GENERATION FUNCTION ---

def compare_speech(text):
    print(f"\n🎤 Sentence: '{text}'")
    
    # Pre-calculate sequence to save time
    seq = text_to_sequence(text, ['english_cleaners2'])[0]
    x = torch.LongTensor(seq).unsqueeze(0)
    x_lengths = torch.tensor([len(seq)])

    # 1. Neutral Version
    print("⏳ Generating Neutral (ODE Steps: 10)...")
    with torch.no_grad():
        output_n = model_neutral.synthesise(x, x_lengths, n_timesteps=50, temperature=0.667, length_scale=1.0)
        mel_n = output_n['mel']
        
        # --- FIXED: No more infinite loops! ---
        if mel_n.ndim == 4:
            # If it' [1, 80, 80, Time], we just take the first '80' 
            mel_n = mel_n[:, 0, :, :] 
            print(mel_n.shape)
            
        audio_n = vocoder(mel_n)
    
    # 2. Angry Version
    print("⏳ Generating Angry (ODE Steps: 10)...")
    with torch.no_grad():
        output_a = model_angry.synthesise(x, x_lengths, n_timesteps=50, temperature=0.667, length_scale=0.8)
        mel_a = output_a['mel']
        
        # --- FIXED ---
        if mel_a.ndim == 4:
            mel_a = mel_a[:, 0, :, :]
            print(mel_n.shape)
            
        audio_a = vocoder(mel_a)

    print("\n🔊 NEUTRAL (Baseline):")
    display(Audio(audio_n.squeeze().numpy(), rate=22050))
    
    print("🔊 ANGRY (3,000 Epochs):")
    display(Audio(audio_a.squeeze().numpy(), rate=22050))

# --- 4. RUN THE COMPARISON ---
compare_speech("I am absolutely finished with these architecture errors!")

📦 Loading Neutral Model (Vocab size: 178)...
📦 Loading Fine-tuned Angry Model (Vocab size: 198)...

🎤 Sentence: 'I am absolutely finished with these architecture errors!'
⏳ Generating Neutral (ODE Steps: 10)...
torch.Size([1, 80, 216])
⏳ Generating Angry (ODE Steps: 10)...
torch.Size([1, 80, 216])

🔊 NEUTRAL (Baseline):


🔊 ANGRY (3,000 Epochs):


In [34]:
from matcha.text import text_to_sequence
from matcha.text.symbols import symbols

text = "Hello"
# Get the IDs and the cleaned text
ids, cleaned_text = text_to_sequence(text, ['english_cleaners2'])

print(f"📝 Original: {text}")
print(f"🧹 Cleaned IPA: {cleaned_text}")
print(f"🔢 IDs sent to model: {ids}")

# Reconstruct the text from the IDs to see if the dictionary matches
reconstructed = "".join([symbols[i] for i in ids if i < len(symbols)])
print(f"🗣️ The Model thinks you said: {reconstructed}")

📝 Original: Hello
🧹 Cleaned IPA: həlˈoʊ
🔢 IDs sent to model: [50, 83, 54, 156, 57, 135]
🗣️ The Model thinks you said: həlˈoʊ


# Baseline Normal

In [54]:
import torch
import datetime as dt
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import IPython.display as ipd

# --- 1. PRODUCING THE TEXT INPUTS ---
@torch.inference_mode()
def process_text(text: str):
    # 'intersperse' adds blank tokens (0) between phonemes for better rhythm
    seq = text_to_sequence(text, ['english_cleaners2'])[0]
    x = torch.tensor(intersperse(seq, 0), dtype=torch.long)[None] 
    
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    
    # We'll use the symbols to see what it's saying
    from matcha.text.symbols import symbols
    x_phones = "".join([symbols[i] for i in x.squeeze(0).tolist() if i < len(symbols)])
    
    return {
        'x_orig': text,
        'x': x,
        'x_lengths': x_lengths,
        'x_phones': x_phones
    }

# --- 2. THE WAVEFORM CONVERTER ---
# Adding a simple denoiser logic since HiFi-GAN can be 'hissy'
def to_waveform(mel, vocoder):
    with torch.no_grad():
        # Ensure mel is [1, 80, T]
        if mel.ndim == 4: mel = mel.squeeze(1)
        
        audio = vocoder(mel).clamp(-1, 1)
        
        # Note: If your local vocoder doesn't have a built-in denoiser, 
        # we skip the 'strength' part to avoid errors, or use a simple squeeze.
        return audio.cpu().squeeze()

# --- 3. THE SYNTHESIS ENGINE ---
@torch.inference_mode()
def synthesise(text, model, n_timesteps=10, temperature=0.667, length_scale=1.0):
    text_processed = process_text(text)
    start_t = dt.datetime.now()
    
    output = model.synthesise(
        text_processed['x'],
        text_processed['x_lengths'],
        n_timesteps=n_timesteps,
        temperature=temperature,
        spks=None,
        length_scale=length_scale
    )
    
    output.update({'start_t': start_t, **text_processed})
    return output

In [55]:
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.config import v1
from matcha.utils.utils import intersperse
import json

# Use the checkpoint you have in your local folder
HIFIGAN_CHECKPOINT = r"C:\Users\alvar\Code\Github\MedEmoji-TTS\model\hifigan_T2_v1"

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self

def load_vocoder(checkpoint_path):
    h = AttrDict(v1)
    device = torch.device("cpu")
    hifigan = HiFiGAN(h).to(device)
    # We load the 'generator' key from the checkpoint
    state_dict = torch.load(checkpoint_path, map_location=device)
    if 'generator' in state_dict:
        hifigan.load_state_dict(state_dict['generator'])
    else:
        hifigan.load_state_dict(state_dict)
    hifigan.eval()
    hifigan.remove_weight_norm()
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
denoiser = Denoiser(vocoder)
print("🎸 Local Vocoder + Denoiser Loaded!")

C:\Users\alvar\AppData\Local\Temp\ipykernel_5676\2027618891.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=device

Removing weight norm...
🎸 Local Vocoder + Denoiser Loaded!


In [56]:
@torch.inference_mode()
def to_waveform(mel, vocoder, denoiser):
    # Fix the 4D broadcasting error [1, 80, 80, T] -> [1, 80, T]
    if mel.ndim == 4:
        if mel.shape[1] == mel.shape[2] == 80:
            mel = mel[:, 0, :, :] 
        else:
            mel = mel.squeeze(1)

    # Convert to audio
    audio = vocoder(mel).clamp(-1, 1)
    
    # Apply Denoiser (This stops the 'Robotic' metallic sound)
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    
    return audio

In [57]:
def run_synthesis(text, model_to_use, name):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=10, 
            temperature=0.667, 
            length_scale=1.0
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

# Run the test
run_synthesis("I am absolutely finished with these architecture errors!", model_neutral, "Neutral Base")

🎬 Testing: Neutral Base


In [61]:
def run_synthesis(text, model_to_use, name):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=10, 
            temperature=0.667, 
            length_scale=1.0
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

# Run the test
run_synthesis("I am absolutely finished with these architecture errors!", model_angry, "Neutral Base")

🎬 Testing: Neutral Base
